# Человек в цикле: преддействия, ранжирование рисков и аудит логов

В README к этому уроку введено понятие Человека в цикле с коротким примером, который предлагает пользователю выбрать `APPROVE` (одобрить) или `REJECT` (отклонить) после того, как агент уже дал ответ. Этот паттерн — хорошая отправная точка, но в производственных реализациях HITL обычно требуются три дополнительных элемента:

1. **Преддействие (pre-action gate)**, которое срабатывает **до** того, как агент выполнит рискованный шаг, чтобы контролировать затраты, необратимость и задержки.
2. **Ранжирование рисков (risk tiering)**, чтобы действия с низким риском выполнялись автоматически, действия со средним риском подтверждались пакетно, а действия с высоким риском требовали вмешательства человека.
3. **Аудит лог и цикл доработки**, чтобы каждое решение преддействия сохранялось в формате JSONL, а при отклонении агент получал структурированное описание причины для переобработки вместо простого вывода `Revising...`.

В этой тетрадке каждый из этих элементов построен на тех же примитивах, что и `06-system-message-framework.ipynb`. Она запускается от начала до конца в режиме `DEMO_MODE = True` (ввод с клавиатуры не требуется) или с реальными вызовами `input()`, если `DEMO_MODE = False`. Примечание: в DEMO_MODE циклы доработки по третьей цели прописаны сценарно, чтобы была видна вся механика. Для реальной переоценки с доработкой нужен `DEMO_MODE = False` и оператор.

**Вне области рассмотрения (освещается в других уроках):** аутентификация и контроль доступа (угроза 2 в README урока 06), промежуточное ПО для вызова инструментов (глубокое погружение в MAF урока 14), паттерны многоагентных дискуссий.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Шаблон 1: Ворота перед действием

Фрагмент HITL из README сначала вызывает агента, а затем просит пользователя одобрить результат. Это поток **после действия**. Агент уже выполнился, поэтому стоимость вызова LLM уже оплачена, и любой побочный эффект (отправленное письмо, записанная строка в базе данных, размещённый комментарий) уже произошёл.

Поток **до действия** вставляет ворота перед выполнением рискованного шага агентом. Агент предлагает действие, ворота решают, выполнять ли его, и только после одобрения происходит побочный эффект.

| Аспект | Одобрение после действия (фрагмент из README) | Ворота перед действием (этот ноутбук) |
|---|---|---|
| Когда выполняется одобрение? | После того, как агент сгенерировал результат | До выполнения любого побочного эффекта |
| Расходы на LLM при отклонении | Уже оплачены | Оплачивается только предложение, а не действие |
| Необратимые побочные эффекты | Возможны (действие уже произошло) | Предотвращаются |
| Прозрачность аудита | Одобрение — это оператор вывода | Одобрение — это запись в JSON с меткой времени, действием, причиной |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Шаблон 2: Уровни риска

Не каждое действие требует одобрения человеком. Запрос только для чтения через публичное API имеет одни риски, а отправка клиенту письма — другие. Одинаковое обращение с этими случаями тратит внимание оператора и замедляет агента.

Простая модель из 3 уровней:

| Уровень | Примеры | Процесс одобрения |
|---|---|---|
| `низкий` (только чтение) | Поиск в базе знаний, просмотр вариантов рейсов, получение публичной веб-страницы | Автоматическое выполнение, записывается для аудита |
| `средний` (легкое изменение) | Кэширование результата, увеличение счетчика, планирование напоминания | Автоматическое выполнение, но с ежедневным обзором |
| `высокий` (внешнее воздействие или необратимые действия) | Отправка письма, списание с карты, публикация в публичном канале | Блокировка до одобрения человеком |

Это один из вариантов уровней. Производственные системы часто используют более детализированные уровни (например, уровни прав AWS IAM, уровни доступа на основе ролей). Ниже приведенная версия с 3 уровнями — это минимально полезный вариант для агента, совмещающего действия только для чтения и с побочными эффектами.

Классификатор ниже использует эвристику ключевых слов, чтобы демонстрация оставалась детерминированной и дешевой. В производственной системе вы бы заменили его на обученный классификатор или движок политик.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Шаблон 3: Журнал аудита и цикл доработки

`print("Response approved.")` — это не журнал аудита. Для доверия каждое решение на вратарном уровне должно фиксироваться как структурированное событие, к которому позже можно выполнить запрос, воспроизведение или приложить к разбору инцидента.

Две части:

1. **JSONL только для добавления.** Одна строка на решение с отметкой времени, действием, уровнем, решением, причиной. Легко искать, легко позже отправить в настоящий хранилище логов.
2. **Цикл доработки при отклонении.** Когда вратарь возвращает `deny`, агент заново предлагает запрос с причиной отклонения в контексте, чтобы следующее предложение могло избежать проблемы.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Дополнительные ресурсы

Несколько других публичных проектов реализуют вариации этих паттернов HITL. Сравните подходы, чтобы найти подходящий для вашего стека:

- **LangChain** обертки инструментов с живым участием человека ([документация](https://python.langchain.com/docs/integrations/tools/human_tools)): обертки инструментов, которые приостанавливают выполнение для ввода человеком.
- **AutoGen** `UserProxyAgent` ([документация v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); в AutoGen v0.4+ изменена структура): использует роль агента специально для представления человека в многоагентных обсуждениях.
- **Microsoft Agent Framework (MAF)** промежуточное программное обеспечение для вызова функций ([документация](https://learn.microsoft.com/agent-framework/)): промежуточное ПО, которое запускается вокруг каждого вызова инструмента/функции, подходит для реализации логики допуска и потоков утверждения.

Каждый проект по-разному обрабатывает три подпаттерна: LangChain оборачивает их как инструменты, AutoGen использует роль агента, а Microsoft Agent Framework применяет промежуточное ПО для вызова функций. Ознакомьтесь с одной-двумя реализациями целиком, прежде чем выбирать дизайн для собственного агента.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
